# Ensamble simple: ¿ayuda promediar varios modelos?

## Qué responde este notebook

Si varios modelos cometen errores distintos, promediar sus predicciones puede reducir el error sin
entrenar nada nuevo. Eso solo funciona si los modelos son **diversos**: promediar modelos casi
idénticos no reduce nada. Aquí se mide esa diversidad y se prueban ensambles contra los modelos
elegidos en [`4.4_ajuste_hiperparametros.ipynb`](4.4_ajuste_hiperparametros.ipynb), con la misma
regla que el resto de la etapa: se adopta un ensamble solo si mejora la métrica principal en
validación y la mejora es real (el intervalo bootstrap por semanas no incluye cero). La prueba solo
se reporta.

In [1]:
import sys
sys.path.insert(0, "../src")
import data, features, modeling, experimentos

import numpy as np
import pandas as pd
from xgboost import XGBRegressor

pd.set_option("display.width", 160)

AZUL, NARANJA, VERDE, GRIS, GRIS_CLARO = "#2a78d6", "#eb6834", "#1baf7a", "#888780", "#c3c2b7"

wr = features.construir_tabla_modelado(range(2016, 2026))
features_arbol = features.columnas_por_modelo("arbol")
features_ridge = features.columnas_por_modelo("ridge")
train_mask, val_mask, test_mask = experimentos.dividir_temporal(wr, range(2016, 2022), [2022, 2023], [2024, 2025])

CONFIG_ELEGIDA = {
    "receptions": dict(max_depth=3, n_estimators=200, learning_rate=0.03),
    "receiving_yards": dict(max_depth=3, n_estimators=200, learning_rate=0.03),
    "receiving_tds": dict(objective="count:poisson", max_depth=3, n_estimators=200, learning_rate=0.08),
}

## 1. Un caso concreto antes que cualquier tabla

Ja'Marr Chase, el receptor de [`2.6_eda_jugador_destacado.ipynb`](2.6_eda_jugador_destacado.ipynb), en su
última semana de 2025: ¿qué tan distintas son, para el mismo partido, las predicciones de yardas de
los cuatro modelos de [`4.2_modelos_recepciones_yardas.ipynb`](4.2_modelos_recepciones_yardas.ipynb)? Son XGBoost, RandomForest,
HistGradientBoosting y Ridge, cada uno con los hiperparámetros que eligió en validación.

In [2]:
resultados_yardas, resumen_yardas = experimentos.entrenar_y_evaluar(
    wr, "receiving_yards", features_arbol, features_ridge, train_mask, val_mask, test_mask
)
chase_id = data.cargar_jugadores().pipe(lambda j: j[j["display_name"] == "Ja'Marr Chase"])["gsis_id"].iloc[0]
idx_chase = wr[test_mask & (wr["player_id"] == chase_id)].sort_values(["season", "week"]).index[-1]
print(f"real: {wr.loc[idx_chase, 'receiving_yards']:.0f} yardas")
for nombre in ["XGBoost", "RandomForest", "HistGradientBoosting", "Ridge"]:
    print(f"  {nombre}: {resultados_yardas[nombre]['pred'][idx_chase]:.1f}")

real: 96 yardas
  XGBoost: 87.2
  RandomForest: 84.0
  HistGradientBoosting: 80.7
  Ridge: 85.5


**Lectura.** Hizo 96 yardas y los cuatro modelos predicen entre 80.7 y 87.2: los cuatro se quedan
cortos, por montos parecidos. Su promedio cae dentro de ese mismo rango, así que tampoco se acerca a
96; aquí queda incluso más lejos que el XGBoost solo. Un partido no basta para concluir nada; la
sección 2 mide qué tan parecidas son las predicciones en toda la validación.

## 2. La diversidad, medida con la correlación de las predicciones de yardas en validación

In [3]:
def correlacion_predicciones(resultados, nombres, mascara):
    return pd.DataFrame({n: resultados[n]["pred"][mascara] for n in nombres}).corr()

correlacion_predicciones(resultados_yardas, ["XGBoost", "RandomForest", "HistGradientBoosting", "Ridge"], val_mask).round(3)

,XGBoost,RandomForest,HistGradientBoosting,Ridge
XGBoost,1.000,0.990,0.985,0.976
RandomForest,0.990,1.000,0.979,0.976
HistGradientBoosting,0.985,0.979,1.000,0.961
Ridge,0.976,0.976,0.961,1.000


**Lectura.** Los modelos de árboles se parecen mucho entre sí: correlación de 0.979 a 0.990 en
validación.
Ridge es el más distinto, y aun así correlaciona 0.961-0.976 con los demás. Un promedio solo
reduce el error cuando los modelos se equivocan en partidos distintos; con correlaciones así de
altas, se espera poca ganancia.

## 3. Recepciones y yardas: ensambles contra el modelo elegido

Dos ensambles por resultado, con pesos iguales: los tres modelos de árboles, y los tres árboles más
Ridge. Se comparan contra el XGBoost elegido en RMSE de validación; la prueba se muestra al lado.

In [4]:
resultados_recepciones, _ = experimentos.entrenar_y_evaluar(
    wr, "receptions", features_arbol, features_ridge, train_mask, val_mask, test_mask
)

def evaluar_ensambles(target, resultados):
    media = wr.loc[train_mask, target].mean()
    p = {n: resultados[n]["pred"] for n in ("XGBoost", "RandomForest", "HistGradientBoosting", "Ridge")}
    versiones = {
        "XGBoost elegido": p["XGBoost"],
        "ensamble árboles (3)": (p["XGBoost"] + p["RandomForest"] + p["HistGradientBoosting"]) / 3,
        "ensamble árboles + Ridge (4)": (p["XGBoost"] + p["RandomForest"] + p["HistGradientBoosting"] + p["Ridge"]) / 4,
    }
    filas = {}
    for nombre, pred in versiones.items():
        pred = pd.Series(np.clip(pred, 0, None), index=wr.index)
        val = modeling.metricas_target(target, wr.loc[val_mask, target], pred[val_mask], media)
        test = modeling.metricas_target(target, wr.loc[test_mask, target], pred[test_mask], media)
        fila = {"val_rmse": val["rmse"], "val_r2_oos": val["r2_oos"], "test_rmse": test["rmse"], "test_r2_oos": test["r2_oos"]}
        if nombre != "XGBoost elegido":
            c = experimentos.diferencia_bootstrap(wr[val_mask], wr.loc[val_mask, target], pred[val_mask],
                                                  versiones["XGBoost elegido"][val_mask], "rmse")
            fila.update({"dif_val_vs_elegido": c["diferencia"], "IC_inf": c["ic_inferior"], "IC_sup": c["ic_superior"],
                         "se_adopta": (not c["empate"]) and c["diferencia"] < 0})
        filas[nombre] = fila
    return pd.DataFrame(filas).T.infer_objects()

for target, resultados in (("receiving_yards", resultados_yardas), ("receptions", resultados_recepciones)):
    params = resultados["XGBoost"]["modelo"].get_params()
    print(target, "| XGBoost de la familia:", {k: params[k] for k in ("max_depth", "n_estimators", "learning_rate")},
          "| configuracion elegida en 4.4:", CONFIG_ELEGIDA[target])
    print(evaluar_ensambles(target, resultados).to_string(float_format=lambda x: f"{x:.4f}"))
    print()

receiving_yards | XGBoost de la familia: {'max_depth': 3, 'n_estimators': 200, 'learning_rate': 0.03} | configuracion elegida en 4.4: {'max_depth': 3, 'n_estimators': 200, 'learning_rate': 0.03}
                              val_rmse  val_r2_oos  test_rmse  test_r2_oos  dif_val_vs_elegido  IC_inf  IC_sup se_adopta
XGBoost elegido                29.5718      0.3624    27.9143       0.3754                 NaN     NaN     NaN       NaN
ensamble árboles (3)           29.5293      0.3643    27.8966       0.3761             -0.0425 -0.0977  0.0075     False
ensamble árboles + Ridge (4)   29.5357      0.3640    27.8470       0.3784             -0.0361 -0.1007  0.0226     False

receptions | XGBoost de la familia: {'max_depth': 3, 'n_estimators': 200, 'learning_rate': 0.03} | configuracion elegida en 4.4: {'max_depth': 3, 'n_estimators': 200, 'learning_rate': 0.03}
                              val_rmse  val_r2_oos  test_rmse  test_r2_oos  dif_val_vs_elegido  IC_inf  IC_sup se_adopta
XGBoost e

**Lectura.** La primera línea de cada bloque confirma que el XGBoost de la familia es la
configuración elegida en 4.4. En los dos resultados, los ensambles quedan apenas por debajo de él en
RMSE de validación (yardas: −0.0425 y −0.0361; recepciones: −0.0022 y −0.0028), pero en los cuatro
casos el intervalo de 95% incluye cero: es un empate y se queda el XGBoost, la regla de 4.1. En
prueba las diferencias son igual de pequeñas: el ensamble de cuatro modelos tiene un RMSE de 27.85
contra 27.91 en yardas y de 1.821 contra 1.826 en recepciones. Son centésimas de yarda y milésimas
de recepción, que no justifican mantener cuatro modelos en el flujo semanal.

## 4. Touchdowns: mezclar el modelo elegido con el de dos etapas

El modelo de dos etapas (*hurdle*) de [`4.3_modelos_touchdowns.ipynb`](4.3_modelos_touchdowns.ipynb)
tiene una lógica distinta (primero clasifica si anota, luego cuenta), así que puede ser más diverso.
Se mezcla con el XGBoost Poisson elegido, `w · XGBoost + (1 − w) · hurdle`, y el peso se elige por
deviance de Poisson en validación.

In [5]:
target = "receiving_tds"
y_tds = wr[target]
media_tds = y_tds[train_mask].mean()
modelo_elegido = XGBRegressor(**CONFIG_ELEGIDA[target], random_state=0, n_jobs=-1)
modelo_elegido.fit(wr.loc[train_mask, features_arbol], y_tds[train_mask])
pred_elegido = pd.Series(np.clip(modelo_elegido.predict(wr[features_arbol]), 0, None), index=wr.index)
salida_hurdle = experimentos.entrenar_hurdle(wr, target, features_arbol, train_mask, val_mask, test_mask)
pred_hurdle = salida_hurdle["pred"].clip(lower=0)

print("correlacion de predicciones en validacion, XGBoost elegido vs hurdle:",
      round(np.corrcoef(pred_elegido[val_mask], pred_hurdle[val_mask])[0, 1], 3))

filas = {}
mezclas = {}
for w in (1.0, 0.7, 0.5, 0.3, 0.0):
    pred = w * pred_elegido + (1 - w) * pred_hurdle
    mezclas[w] = pred
    val = modeling.metricas_target(target, y_tds[val_mask], pred[val_mask], media_tds)
    test = modeling.metricas_target(target, y_tds[test_mask], pred[test_mask], media_tds)
    filas[f"w = {w}"] = {"val_deviance": val["deviance_poisson"], "val_d2_oos": val["d2_oos"], "val_r2_oos": val["r2_oos"],
                         "val_pendiente": val["pendiente_calibracion"], "test_deviance": test["deviance_poisson"], "test_d2_oos": test["d2_oos"]}
tabla_mezcla = pd.DataFrame(filas).T
mejor_w = float(tabla_mezcla["val_deviance"].idxmin().split("= ")[1])
c = experimentos.diferencia_bootstrap(wr[val_mask], y_tds[val_mask], mezclas[mejor_w][val_mask], pred_elegido[val_mask], "deviance_poisson")
print(f"mejor peso en validacion: w = {mejor_w}; diferencia contra el elegido = {c['diferencia']:+.4f} "
      f"(IC 95%: {c['ic_inferior']:+.4f} a {c['ic_superior']:+.4f}) -> {'empate' if c['empate'] else 'diferencia real'}")
tabla_mezcla.round(4)

correlacion de predicciones en validacion, XGBoost elegido vs hurdle: 0.925
mejor peso en validacion: w = 0.7; diferencia contra el elegido = -0.0001 (IC 95%: -0.0018 a +0.0015) -> empate


,val_deviance,val_d2_oos,val_r2_oos,val_pendiente,test_deviance,test_d2_oos
w = 1.0,0.6167,0.1223,0.0826,0.9604,0.6252,0.1355
w = 0.7,0.6165,0.1225,0.0832,0.9686,0.6265,0.1338
w = 0.5,0.6174,0.1213,0.0822,0.9588,0.6283,0.1314
w = 0.3,0.6190,0.1190,0.0800,0.9375,0.6307,0.1280
w = 0.0,0.6228,0.1135,0.0745,0.8873,0.6356,0.1212


**Lectura.** El modelo de dos etapas se parece al elegido más de lo que su lógica distinta haría
esperar: correlación de 0.925 en validación. El mejor peso es w = 0.7, con una deviance de validación de 0.6165 contra 0.6167 del
XGBoost solo: una diferencia de −0.0001 con intervalo de −0.0018 a +0.0015, es decir, un empate (y con
cuatro pesos probados, el intervalo ajustado por comparaciones múltiples sería aún más ancho). Con más
peso para el de dos etapas la deviance empeora hasta 0.6228 (w = 0) y la pendiente de calibración baja
hasta 0.89, fuera del rango 0.9–1.1. En prueba, la mezcla con w = 0.7 queda por detrás del XGBoost
solo (deviance de 0.6265 contra 0.6252). Se queda el XGBoost solo.

## 5. Registro de lo que se probó

Ninguna alternativa se adopta, así que sus modelos no se guardan: se regeneran corriendo este
notebook. Solo queda la metadata de cada prueba (qué se probó, sus métricas y la decisión) en
`weekly/wr/models/exploratorio/`.

In [6]:
import json, os

CARPETA = "../models/exploratorio"
os.makedirs(CARPETA, exist_ok=True)
guardados = []
for target, resultados in (("receptions", resultados_recepciones), ("receiving_yards", resultados_yardas)):
    tabla = evaluar_ensambles(target, resultados)
    metadata = {
        "target": target,
        "ensambles_probados": ["arboles (XGBoost + RandomForest + HistGradientBoosting) / 3", "arboles + Ridge / 4"],
        "metrica": "rmse",
        "validacion_2022_2023": tabla["val_rmse"].round(4).to_dict(),
        "prueba_2024_2025": tabla["test_rmse"].round(4).to_dict(),
        "se_adopta": bool(tabla["se_adopta"].eq(True).any()),
        "fecha_de_guardado": pd.Timestamp.now(tz="UTC").isoformat(),
        "features_arbol_en_orden": features_arbol,
        "features_ridge_en_orden": features_ridge,
    }
    with open(f"{CARPETA}/{target}_ensamble_metadata.json", "w") as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False, default=float)
    guardados.append(f"{target}_ensamble_metadata.json")

metadata_hurdle = {
    "target": "receiving_tds",
    "mezcla": "w * XGBoost elegido + (1 - w) * hurdle",
    "metrica": "deviance_poisson",
    "barrido": tabla_mezcla.round(4).to_dict(orient="index"),
    "mejor_w_validacion": mejor_w,
    "se_adopta": (not c["empate"]) and c["diferencia"] < 0,
    "fecha_de_guardado": pd.Timestamp.now(tz="UTC").isoformat(),
    "features_arbol_en_orden": features_arbol,
}
with open(f"{CARPETA}/receiving_tds_hurdle_metadata.json", "w") as f:
    json.dump(metadata_hurdle, f, indent=2, ensure_ascii=False, default=float)
guardados.append("receiving_tds_hurdle_metadata.json")
print("guardados en models/exploratorio:", guardados)

guardados en models/exploratorio: ['receptions_ensamble_metadata.json', 'receiving_yards_ensamble_metadata.json', 'receiving_tds_hurdle_metadata.json']


## Cierre

Ningún ensamble se adopta:

| Resultado | Mejor alternativa | Diferencia en validación contra el XGBoost elegido | Decisión |
|---|---|---|---|
| `receiving_yards` | árboles (3) | RMSE −0.0425 (IC 95%: −0.0977 a +0.0075) | empate, se queda XGBoost |
| `receptions` | árboles + Ridge (4) | RMSE −0.0028 (IC 95%: −0.0083 a +0.0022) | empate, se queda XGBoost |
| `receiving_tds` | mezcla con dos etapas (w = 0.7) | deviance −0.0001 (IC 95%: −0.0018 a +0.0015) | empate, se queda XGBoost Poisson |

Los modelos de árboles son demasiado parecidos para que promediarlos ayude, y en touchdowns la mezcla con el
modelo de dos etapas empata con el XGBoost Poisson solo. Sus modelos no se guardan;
en `weekly/wr/models/exploratorio/` queda solo la metadata de cada prueba.

Anterior: [`4.5_regresion_cuantiles.ipynb`](4.5_regresion_cuantiles.ipynb) · Siguiente:
[`4.7_jerarquico_touchdowns.ipynb`](4.7_jerarquico_touchdowns.ipynb)